In [0]:
%sql
use catalog workspace;
use schema customer360;

select current_catalog(), current_schema();

select * from customer360.customers_clean;

# Load Delta table- customer_clean 

In [0]:
dfCustomers = spark.table(
    "workspace.customer360.customers_clean"
)

display(dfCustomers)

# Create Accounts table

In [0]:
%python
accounts_data = [
        (5001, 1001, "Wireless", "Active", 120.50),
        (5002, 1001, "Internet", "Active", 75.00),
        (5003, 1002, "Fiber", "Active", 145.99),
        (5004, 1003, "Wireless", "Suspended", 85.25),
        (5005, 1006, "Fiber", "Active", 110.00)
]

accounts_column = [    
                   "acctId",
                    "custId",
                    "ServiceType",
                    "AccountStatus",
                    "Balance"]
dfAccounts = spark.createDataFrame(accounts_data, accounts_column)
dfAccounts.write.mode("overwrite").saveAsTable("accounts")

display(dfAccounts)

# **Types of Joins**
- Inner Join
- Left Join
- Full outer join
- Right Outer join
- Left Anti Join
- Left Semi join


## INNER JOIN
matching rows of both the tables.

In [0]:
%sql
select c.custId, acctId, CustName, product, ServiceType,accounTStatus
from customer360.customers_clean c
INNER JOIN customer360.accounts a on c.custid = a.custId

## LEFT JOIN
All the rows of the LEFT table(customers) and matching rows of the RIGHT table(accounts)

In [0]:
%sql
select c.custId, c.CustName, c.product, a.acctId, a.custId, a.ServiceType, a.accountStatus
from customer360.customers_clean c
LEFT JOIN customer360.accounts a on c.custId = a.custId

In [0]:
dfLeftJoin = dfCustomers.join(
    dfAccounts,
    dfCustomers.custId == dfAccounts.custId,
    how="left"
)

# display(dfLeftJoin) -- this displays all teh columns from both table

display(dfLeftJoin.select(
    dfCustomers.custId,
    dfCustomers.custName,
    dfCustomers.product,
    dfAccounts.acctId,
    dfAccounts.custId.alias("accountCustId"),
    dfAccounts.ServiceType,
    dfAccounts.AccountStatus
))

## FULL OUTER JOIN

In [0]:
%sql
select c.custId, c.CustName, c.product, a.acctId, a.custId, a.ServiceType, a.accountStatus
from customer360.customers_clean c
FULL JOIN customer360.accounts a on c.custId = a.custId

## LEFT ANTI JOIN

similar to SQL - **NOT EXISTS** cmd. 
fetches all the rows from left table (customers) that **_do not exists in Right_** table(Accounts)

In [0]:
dfLeftAnti = dfCustomers.join(
    dfAccounts,
    dfCustomers.custId == dfAccounts.custId,
    how="leftanti"
)

# display(dfLeftAnti)  -- this displays Only Left table columns

display(dfLeftAnti.select(
    dfCustomers.custId,
    dfCustomers.custName,
    dfCustomers.product
))

### LEFT ANTI can be achieved using sql query--> 
- (1) using NOT EXISTS 
- (2) left join using custId is null

In [0]:
%sql
select c.custId, c.CustName, c.product
from customer360.customers_clean c
WHERE NOT EXISTS (SELECT 1 FROM customer360.accounts a WHERE c.custId = a.custId);

-- #2 Can be achieved using LEFT JOIN with a.custId IS NULL
select c.custId, c.CustName, c.product
from customer360.customers_clean c
LEFT JOIN customer360.accounts a on c.custId = a.custId 
where a.custId is null;

## LEFT SEMI JOIN --> left_semi
it give all the rows/columns of the left DataFrame for which a match exists in right DataFrame but it returns _**ONLY COLUMNS FROM THE LEFT DATAFRAME.**
NOTE- no columns of right table (accounts) is displayed. 
Similar to **WHERE EXISTS()** in SQL

### INNER join vs LEFT SEMI
- _INNER JOIN_ --> column/data from BOTH table
- _LEFT SEMI_ --> rows/columns from** LEFT SIDE ONLY** where a **MATCH EXISTS**

In [0]:
%python
dfCustomers = spark.table(
    "workspace.customer360.customers_clean"
)

display(dfCustomers)

dfAccounts = spark.table(
    "workspace.customer360.accounts"
)

display(dfAccounts)


dfLeftSemi = dfCustomers.join(
    dfAccounts,
    dfCustomers.custId == dfAccounts.custId,
    how="left_Semi"
    )

display(dfLeftSemi)

## The duplicate-column problem
use **alias** for (1) tablename  (2) ColumnName

In [0]:
c = dfCustomers.alias("c")
a = dfAccounts.alias("a")

dfAlias = c.join(
    a,
    c.custId == a.custId,
    how="inner"
)

#display(dfAlias)

display(dfAlias.select(
                        c.custId,
                        c.custName,
                        c.city,
                        c.product,
                        a.acctId,
                        a.custId.alias("acctCustId"),
                        a.ServiceType,
                        a.AccountStatus,
                        a.Balance
                    )
        )

## Aggregation
- sum(balance)
- count()
- avg()

In [0]:
%sql
select c.custId,  sum(a.balance) as TotalBalance
from customer360.customers_clean c
inner join customer360.accounts a on c.custId = a.custId
group by c.custId
order by c.custId

### What is the total account balance for each customer?

In [0]:
%python
from pyspark.sql.functions import *

c = dfCustomers.alias("c")
a = dfAccounts.alias("a")

dfTotalBalance = c.join(
    a,
    c.custId == a.custId,
    how="inner"
)

display(dfTotalBalance.groupBy(
    c.custId
    # , c.custName, 
    #c.product, 
    #a.ServiceType
).agg(
    sum(a.Balance).alias("TotalBalance")
).orderBy(c.custId))

### Aggregations
- What is the total account balance for each customer? --> **groupBy(), sum()**
- How many accounts does each customer have? --> **count()**
- What is the average account balance for each customer? --> **avg()**



In [0]:
%sql
select c.custId, count(*) as TotalAcctPerCust, avg(a.balance) as AvgBalancePerCust, sum(a.balance) as TotalBalancePerCust
from customer360.customers_clean c 
inner join customer360.accounts a on c.custId = a.custId
group by c.custId
order by c.custId

In [0]:
%python
c = dfCustomers.alias("c")
a = dfAccounts.alias("a")

dfAcctCount = c.join(
    a,
    c.custId == a.custId,
    how="inner"
)

display(dfAcctCount.groupBy(
    c.custId
    # , c.custName 
    # , c.product
    # , a.ServiceType
    ) .agg(count(lit(1)).alias("TotalAcctPerCust")
      # , .agg(count(a.custId).alias("TotalAcctPerCust")
      , avg(a.Balance).alias("AvgBalancePerCust")
      , sum(a.Balance).alias("TotalBalancePerCust")
          ).orderBy(c.custId)
        
        )

In [0]:
%python
from pyspark.sql.functions import *

c = dfCustomers.alias("c")
a = dfAccounts.alias("a")

dfCustAcctSummary = c.join(
    a,
    c.custId == a.custId,
    how="full_outer"
)

display(dfCustAcctSummary.groupBy(c.custId, 
                                  c.custName, 
                                  a.custId.alias("AcctCustId")
        ).agg(count(lit(1)).alias("TotalAcctPerCust"),
               avg(a.Balance).alias("AvgBalancePerCust"),
              sum(a.Balance).alias("TotalBalancePerCust")
).orderBy(c.custId)
)

In [0]:
%sql
select 
    c.custId as CustomerID,
    c.custName as CustomerName,
    c.city as City,
    c.state as State,
    c.product as Product,
    coalesce(count(a.acctId), 0) as AccountCount,
    coalesce(sum(a.balance), 0) as TotalBalance,
    coalesce(avg(a.balance), 0) as AverageBalance
from customer360.customers_clean c 
full outer join customer360.accounts a on c.custId = a.custId
group by c.custId, c.custName, c.city, c.state, c.product
order by c.custId

In [0]:
%python
from pyspark.sql.functions import *

dfCustomers = spark.table(
    "workspace.customer360.customers_clean"
)

display(dfCustomers)

dfAccounts = spark.table(
    "workspace.customer360.accounts"
)

# Load tables and create aliases
c = dfCustomers.alias("c")
a = dfAccounts.alias("a")

# Perform full outer join
dfCustAcctSummary = c.join(
    a,
    c.custId == a.custId,
    how="left"
)

# Group by and aggregate with coalesce for NULL handling
customer_account_summary = (
    dfCustAcctSummary.groupBy(
        c.custId,
        c.custName,
        c.city,
        c.state,
        c.product
    ).agg(
        coalesce(count(a.acctId), lit(0)).alias("AccountCount"),
        coalesce(sum(a.Balance), lit(0)).alias("TotalBalance"),
        coalesce(avg(a.Balance), lit(0)).alias("AverageBalance")
    ).select(
        c.custId,
        c.custName,
        c.city,
        c.state,
        c.product,
        col("AccountCount"),
        col("TotalBalance"),
        col("AverageBalance")
    ).orderBy("custId")
)
customer_account_summary.write.mode("overwrite").saveAsTable("customer_account_summary")

display(customer_account_summary)



In [0]:
%sql
select * from customer360.customer_account_summary;

In [0]:
%sql
Select a.*
FROM customer360.accounts a
where a.custid not in (select c.custId from customer360.customer_account_summary c  )



In [0]:
%python

c = dfCustomers.alias("c")
a = dfAccounts.alias("a")

dfOrphanAcct = a.join(
    c,
    c.custId == a.custId,
    how="left_anti"
)

display(dfOrphanAcct)
       


# NOTE --------------------------------------------------->

- Keep all parent/master records → **left**
- Keep only matched records → **inner**
- Find missing children/parents → **left_anti**
- Reconcile both systems completely → **full_outer**